In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import os
import dask
import dask.dataframe as dd
import itertools
from itertools import chain
from math import sqrt, floor, ceil, isnan
import multiprocess
import multiprocessing
import importlib
from importlib import reload
from collections import Counter
from fuzzywuzzy import process, fuzz
import time
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import statsmodels.api as sm
import warnings
warnings.filterwarnings("error")

pd.options.display.max_columns = 500
pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 400

# A customized winsorisation function that handles None values correctly
# The percentiles are taken and winsorisation are done on non-None values only
def winsor2(series,cutoffs):

    import numpy as np
    import scipy as sp
    
    IsNone = np.isnan(series).copy()
    IsNotNone = np.logical_not(IsNone).copy()
    series_NotNonePart = sp.stats.mstats.winsorize(series[IsNotNone],limits=(cutoffs[0],cutoffs[1]))
    series_new = series.copy()
    series_new[IsNone] = np.nan
    series_new[IsNotNone] = series_NotNonePart

    return series_new


In [20]:
market_share_all_markets_byCSA = pd.read_csv('../CleanData/SDC/1A_market_share_all_markets_byCSA.csv')

# By Each CSA, each year, find the number of new underwriters and their market share

new_entry = []

CSAs = list(market_share_all_markets_byCSA['CSA Code'].unique())
for CSA in CSAs:
    market_share_oneCSA = market_share_all_markets_byCSA[market_share_all_markets_byCSA['CSA Code']==CSA].copy()
    years = sorted(list(market_share_oneCSA['calendar_year'].unique()))
    if len(years)>1:
        for year in years[1:]:
            # All prior years' data
            market_share_oneCSA_prior = market_share_oneCSA[market_share_oneCSA['calendar_year']<year]
            market_share_oneCSA_priorbanks = list(market_share_oneCSA_prior['parent_name'].unique())
            market_share_oneCSA_current = market_share_oneCSA[market_share_oneCSA['calendar_year']==year]
            n_current_deals = np.sum(market_share_oneCSA_current['N_deals'])
            market_share_oneCSA_newbanks = \
                market_share_oneCSA_current[~market_share_oneCSA_current['parent_name'].isin(market_share_oneCSA_priorbanks)]
            n_newbanks = len(market_share_oneCSA_newbanks)
            market_share_N_newbanks = np.sum(market_share_oneCSA_newbanks['market_share_N'])

            new_entry = new_entry+[{
                'CSA Code':CSA,
                'calendar_year':year,
                'n_current_deals':n_current_deals,
                'n_newbanks':n_newbanks,
                'market_share_N_newbanks':market_share_N_newbanks,
                }]

new_entry = pd.DataFrame(new_entry)

In [23]:
new_entry.to_parquet("../CleanData/SDC/0K_EntryStats.parquet")
